In [1]:
import os
from pydantic_ai import Agent
from pydantic_ai.capabilities import MCP
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.providers.openai import OpenAIProvider
from rich.console import Console
from rich.markdown import Markdown

## Let us go one step higher in the abstraction by using Agents and letting them decide which tools to call rather than specifying them manually

In [2]:
agent = Agent(
    name="hpc-docs-agent",
    model=OpenAIResponsesModel(
        model_name="@vertexai/gemini-3.5-flash",
        provider=OpenAIProvider(
            base_url="https://ai-gateway.apps.cloud.rt.nyu.edu/v1/",
            api_key=os.getenv("PORTKEY_API_KEY"),
        ),
    ),
    instructions="Be concise and answer questions from retreived knowledge with tools.",
    capabilities=[
        MCP(
            url="https://mcp-gateway.apps.cloud.rt.nyu.edu/rts-docs-algolia-public/mcp",
            id="rts-docs-algolia-mcp",
            headers={
                "x-portkey-api-key": os.getenv("PORTKEY_API_KEY"),
            },
        ),
    ],
)

In [3]:
result = await agent.run("Login to HPC cluster from off campus") # await is added because the agent is run asynchronously

In [4]:
console = Console()
console.print(Markdown(result.output))

To log into the NYU HPC cluster from off-campus, follow these steps:                                               

Step 1: Connect to the NYU VPN                                                                                     

Because you are outside the secure NYU network, you must first establish a VPN connection:                         

 • Set up and connect to the NYU VPN on your local machine.                                                        
 • Linux users: You can use Cisco AnyConnect or the command line with OpenConnect:                                 
                                                                                                                   
    sudo openconnect -b vpn.nyu.edu                                                                                
                                                                                                                   
   (When prompted, log in with your NetID, password, and Duo push/SMS authentication).                             

-------------------------------------------------------------------------------------------------------------------

Step 2: Log in via SSH                                                                                             

Once the VPN connection is active, open your local terminal (or an SSH client like PuTTY or MobaXterm) and run:    

                                                                                                                   
 ssh <NetID>@login.torch.hpc.nyu.edu                                                                               
                                                                                                                   

(Replace login.torch.hpc.nyu.edu with your specific cluster's login hostname if accessing a different resource).   

-------------------------------------------------------------------------------------------------------------------

Step 3: Complete Multi-Factor Authentication (MFA)                                                                 

If you are logging into Torch, you will be prompted to authenticate through Microsoft and Duo:                     

 1 Open ]8;id=6544314;https://microsoft.com/devicelogin\microsoft.com/devicelogin]8;;\ in your web browser.                                                             
 2 Enter the one-time PIN displayed in your terminal.                                                              
 3 Log in with your <NetID>@nyu.edu account and complete the Duo MFA prompt.                                       
 4 Go back to your terminal and press Enter to finish logging in.

## Let's see how we can evaluate the performance of the Agent on this task by varying the LLM used. Here we check if the output from the Agent contains the specific string `login.torch.hpc.nyu.edu` to ensure that the model did not hallucinate a new login address:

In [5]:
from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import Contains

# Create a dataset with test cases
dataset = Dataset(
    name='login-to-hpc',
    cases=[
        Case(
            name="use-gemini-3.5-flash",
            inputs={
                "query": "Login to HPC cluster from off campus",
                "model": "@vertexai/gemini-3.5-flash"
            },
        ),
        Case(
            name="use-gemini-2.5-flash-lite",
            inputs={
                "query": "Login to HPC cluster from off campus",
                "model": "@vertexai/gemini-2.5-flash-lite"
            },
        ),
    ],
    evaluators=[
        Contains(value='login.torch.hpc.nyu.edu', case_sensitive=True),
    ],
)

async def agent_task(inputs: dict) -> str:
    with agent.override(model=
                        OpenAIResponsesModel(
                            model_name=inputs["model"],
                            provider=OpenAIProvider(
                                base_url="https://ai-gateway.apps.cloud.rt.nyu.edu/v1/",
                                api_key=os.getenv("PORTKEY_API_KEY"),
                                ),
                        )
                       ):
        result = await agent.run(user_prompt=inputs["query"])
        return result.output


# Run the evaluation
report = await dataset.evaluate(agent_task)

# Print the results
report.print()

Output()

           Evaluation Summary: agent_task            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Case ID                   ┃ Assertions ┃ Duration ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━┩
│ use-gemini-3.5-flash      │ ✔          │    10.0s │
├───────────────────────────┼────────────┼──────────┤
│ use-gemini-2.5-flash-lite │ ✗          │     2.7s │
├───────────────────────────┼────────────┼──────────┤
│ Averages                  │ 50.0% ✔    │     6.3s │
└───────────────────────────┴────────────┴──────────┘

## Why did the case with `gemini-2.5-flash-lite` fail? Let's check the output from that run:

In [6]:
console = Console()
console.print(Markdown(report.cases[1].output))

To log in to the HPC cluster from off-campus, you need to connect to the VPN first. Once connected to the VPN, you 
can log in to the burst login node by running ssh <NetID>@burst.                                                   

For visualization purposes, you can use the nvgrid partition. After logging into the burst login node, you can     
request an interactive command line session and then start a VNC server. You will need to set a password for your  
VNC session.                                                                                                       

Once the VNC server is running, you can connect to it from your local machine using a VNC client. If you are       
running graphical applications, use vglrun before the command to ensure they use the GPU.                          

For more detailed instructions, you can refer to the documentation on visualization workstations.

## The older, less capable model misunderstood the prompt and answered the question for a different HPC cluster (Cloud bursting). Can we update the prompt to prevent this?